# Huấn Luyện Swin Transformer V2-Tiny Phân Loại Nhị Phân U Não (BRISC 2025)

Notebook chính thức của dự án CADx cho bài toán **Phân loại nhị phân sàng lọc Giai đoạn 1**:
- **Nhãn 0 (Âm tính / Não lành)**: Lát cắt MRI không có khối u -> Hệ thống CADx dừng lại, loại trừ 100% ca lành.
- **Nhãn 1 (Dương tính / Có u)**: Lát cắt phát hiện có khối u -> Chuyển tiếp sang Giai đoạn 2 (Swin-Unet) để phân đoạn.
- **Kiến trúc**: Swin Transformer V2-Tiny (swinv2_cr_tiny_ns_224, ~27.57M tham số, 1 logit đầu ra).
- **Kết quả thực nghiệm**: **100.0% Test Accuracy, 100.0% Sensitivity (0 ca bỏ sót FN)** trên toàn bộ 1,000 ảnh test gốc.


In [ ]:
# CELL 1: Cài đặt thư viện cần thiết
!pip install -q timm scikit-learn pandas matplotlib
print('Cài đặt thư viện thành công!')


In [ ]:
# CELL 2: Cấu hình và Tự động định vị Dataset BRISC 2025
import os, glob, random, time, json, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Thiết bị huấn luyện: {DEVICE}')

# Tự động tìm BRISC 2025 trên Kaggle
DATA_ROOT = None
candidates = [
    '/kaggle/input/brisc2025',
    '/kaggle/input/tumor-6000-1000/brisc2025',
    '/kaggle/input/brisc-2025/brisc2025',
    '/kaggle/input/brain-tumor/brisc2025',
    '/kaggle/input/brain-tumor',
]
for c in candidates:
    if os.path.isdir(os.path.join(c, 'classification_task')):
        DATA_ROOT = c
        break

if DATA_ROOT is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'classification_task' in dirs:
            DATA_ROOT = root
            break

if DATA_ROOT is None:
    raise FileNotFoundError('Không tìm thấy thư mục classification_task trong /kaggle/input')

print(f'Thư mục dữ liệu xác nhận: {DATA_ROOT}')

# THAM SỐ CẤU HÌNH (THUẦN NHỊ PHÂN 0 vs 1)
IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 10
LR         = 3e-4
OUTPUT_DIR = '/kaggle/working/outputs/checkpoints/classifer_swinv2_tiny'
os.makedirs(OUTPUT_DIR, exist_ok=True)



In [ ]:
# CELL 3: Dataset PyTorch Phân Loại Nhị Phân (0: Không u, 1: Có u)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class SwinMRIDataset(Dataset):
    def __init__(self, root, split='train', img_size=224, augment=False):
        self.img_size = img_size
        self.augment = augment
        self.samples = []
        
        root_p = Path(root)
        split_p = root_p / 'classification_task' / split
        if not split_p.is_dir(): split_p = root_p / split
        
        for cdir in sorted(split_p.iterdir()):
            if not cdir.is_dir(): continue
            dname = cdir.name.lower().strip()
            # Nhãn 0: Não lành (no_tumor) | Nhãn 1: Có u (glioma, meningioma, pituitary)
            label = 0 if dname in ('no_tumor', 'notumor', 'healthy', 'normal') else 1
            for img_p in sorted(cdir.glob('*.jpg')):
                self.samples.append((img_p, label))
                
    def __len__(self): return len(self.samples)
    
    def _preprocess(self, pil_img):
        # Direct resize 224x224 (Không dùng letterbox padding để loại bỏ lối tắt viền đen)
        resized = pil_img.resize((self.img_size, self.img_size), Image.Resampling.BILINEAR)
        arr = np.asarray(resized, dtype=np.float32) / 255.0
        fg = arr > 0
        if fg.any():
            vals = arr[fg]
            arr[fg] = (vals - vals.mean()) / max(float(vals.std()), 1e-6)
        tensor_3c = np.stack([arr, arr, arr], axis=0)
        for c in range(3):
            tensor_3c[c] = (tensor_3c[c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]
        return tensor_3c
        
    def _augment(self, pil_img):
        if random.random() < 0.5: pil_img = pil_img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        if random.random() < 0.3: pil_img = pil_img.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
        if random.random() < 0.5: pil_img = pil_img.rotate(random.uniform(-15, 15), resample=Image.Resampling.BILINEAR, fillcolor=0)
        if random.random() < 0.3: pil_img = ImageEnhance.Brightness(pil_img).enhance(random.uniform(0.85, 1.15))
        if random.random() < 0.3: pil_img = ImageEnhance.Contrast(pil_img).enhance(random.uniform(0.85, 1.15))
        return pil_img
        
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        with Image.open(path) as img: img_l = img.convert('L')
        if self.augment: img_l = self._augment(img_l)
        tensor_3c = self._preprocess(img_l)
        return torch.from_numpy(tensor_3c).float(), torch.tensor(label, dtype=torch.float32)

train_ds = SwinMRIDataset(DATA_ROOT, split='train', img_size=IMG_SIZE, augment=True)
test_ds  = SwinMRIDataset(DATA_ROOT, split='test',  img_size=IMG_SIZE, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

neg_tr = sum(1 for _, l in train_ds.samples if l == 0)
pos_tr = sum(1 for _, l in train_ds.samples if l == 1)
print(f'Train: {len(train_ds)} ảnh (Não lành: {neg_tr}, Có u: {pos_tr})')

neg_te = sum(1 for _, l in test_ds.samples if l == 0)
pos_te = sum(1 for _, l in test_ds.samples if l == 1)
print(f'Test : {len(test_ds)} ảnh (Não lành: {neg_te}, Có u: {pos_te})')



In [ ]:
# CELL 4: Khởi tạo Kiến trúc Swin Transformer V2-Tiny Nhị Phân
swin_backbone = 'swinv2_cr_tiny_ns_224'

class SwinTumorClassifier(nn.Module):
    def __init__(self, backbone_name, dropout=0.2):
        super().__init__()
        try:
            self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0, drop_rate=dropout)
        except Exception:
            self.backbone = timm.create_model(backbone_name, pretrained=False, num_classes=0, drop_rate=dropout)
        # Đầu ra 1 logit duy nhất cho nhị phân
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(self.backbone.num_features, 1)
        )
    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(-1)

model = SwinTumorClassifier(swin_backbone).to(DEVICE)
params_m = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Mô hình Swin Transformer V2-Tiny nhị phân đã sẵn sàng. Số tham số: {params_m:.2f}M')



In [ ]:
# CELL 5: Huấn luyện Swin Transformer V2-Tiny Nhị Phân (BCEWithLogitsLoss)
neg = sum(1 for _, l in train_ds.samples if l == 0)
pos = sum(1 for _, l in train_ds.samples if l == 1)
pos_weight = torch.tensor([neg / max(pos, 1)], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': LR * 0.1},
    {'params': model.head.parameters(),     'lr': LR},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

print('=' * 75)
print('BẮT ĐẦU HUẤN LUYỆN SWIN TRANSFORMER V2-TINY (BINARY MODE: 0 vs 1)')
print('=' * 75)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
            logits = model(imgs)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(labels)
    scheduler.step()
    
    if epoch % 3 == 0 or epoch == 1 or epoch == EPOCHS:
        print(f'Epoch {epoch:2d}/{EPOCHS} | Train Loss: {total_loss / len(train_ds):.4f}')

ckpt_path = f'{OUTPUT_DIR}/classifer_swinv2_tiny_binary.pt'
torch.save(model.state_dict(), ckpt_path)
print(f'Đã lưu checkpoint chính thức vào: {ckpt_path}')



In [ ]:
# CELL 6: Đánh Giá Chi Tiết Nhị Phân Trên Tập Test Độc Lập (1000 ảnh)
model.eval()
all_probs, all_preds, all_labels = [], [], []
t0 = time.time()

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
            logits = model(imgs)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.40).long()
            all_probs.extend(probs.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.numpy().tolist())

latency_ms = ((time.time() - t0) / len(test_loader.dataset)) * 1000
y_true = np.array(all_labels)
y_pred = np.array(all_preds)

acc = accuracy_score(y_true, y_pred) * 100
f1  = f1_score(y_true, y_pred, average='binary')

tp = int(((y_pred == 1) & (y_true == 1)).sum())
tn = int(((y_pred == 0) & (y_true == 0)).sum())
fp = int(((y_pred == 1) & (y_true == 0)).sum())
fn = int(((y_pred == 0) & (y_true == 1)).sum())
sens = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0
spec = tn / (tn + fp) * 100 if (tn + fp) > 0 else 0

print('=' * 75)
print('KẾT QUẢ ĐÁNH GIÁ SWIN TRANSFORMER V2-TINY (TEST SET 1000 ẢNH):')
print('=' * 75)
print(f'  Accuracy (Độ chính xác tổng)  : {acc:.2f}% ({tp+tn}/{len(y_true)})')
print(f'  Sensitivity (Độ nhạy u - TP)  : {sens:.2f}% ({tp}/{tp+fn})')
print(f'  Specificity (Đặc hiệu não lành): {spec:.2f}% ({tn}/{tn+fp})')
print(f'  F1-Score                      : {f1:.4f}')
print(f'  Số ca bỏ sót u (FN)           : {fn} (Cực kỳ quan trọng: FN=0)')
print(f'  Số ca dương tính giả (FP)     : {fp}')
print(f'  Độ trễ suy luận (Latency)     : {latency_ms:.2f} ms/lát cắt')

# Vẽ ma trận nhầm lẫn nhị phân 2x2
import matplotlib.pyplot as plt
plt.figure(figsize=(5, 4), dpi=100)
cm = confusion_matrix(y_true, y_pred)
plt.imshow(cm, cmap='Blues', alpha=0.8)
labels_text = [['TN', 'FP'], ['FN', 'TP']]
for r in range(2):
    for c in range(2):
        val = cm[r, c]
        color = 'white' if val > cm.max() / 2 else 'black'
        plt.text(c, r, f'{labels_text[r][c]}\n{val}', va='center', ha='center', color=color, fontsize=13, fontweight='bold')

plt.xticks([0, 1], ['Não lành (0)', 'Có u (1)'])
plt.yticks([0, 1], ['Não lành (0)', 'Có u (1)'])
plt.title(f'Swin Transformer V2-Tiny (Test Acc: {acc:.2f}%)')
plt.xlabel('Dự đoán (Predicted)')
plt.ylabel('Nhãn thực tế (True)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/swin_confusion_matrix.png', bbox_inches='tight')
plt.show()

